# Demo 03. Cubic Splines and Boundary Conditions

**Standard 3** (splines), Week 3.

A single high-degree polynomial through many points can diverge (demo 01). Splines instead use low-degree pieces joined with prescribed smoothness. This demo constructs piecewise-linear and cubic splines, exhibits the tridiagonal system defining the cubic, and shows the effect of the boundary condition.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## The tridiagonal system

Let $M_i = s''(x_i)$ denote the second derivatives at the nodes and $h_i = x_{i+1}-x_i$. Continuity of the first derivative at each interior node gives, for $i = 1,\dots,n-1$,

$$ h_{i-1} M_{i-1} + 2(h_{i-1}+h_i) M_i + h_i M_{i+1} = 6\left(\frac{y_{i+1}-y_i}{h_i} - \frac{y_i-y_{i-1}}{h_{i-1}}\right). $$

This is a tridiagonal system in the $M_i$, solvable in $O(n)$. It supplies $n-1$ equations for $n+1$ unknowns; the two remaining equations are the boundary conditions, either natural ($M_0=M_n=0$) or clamped (prescribed end slopes).

In [ ]:
# ---------------------------------------------------------------------------
# Piecewise-linear interpolation 
# ---------------------------------------------------------------------------
# The simplest spline: "connect the dots" with straight segments. Continuous,
# but with kinks (its first derivative jumps at the nodes). Cubic splines below
# remove those kinks by matching first and second derivatives too.

def piecewise_linear(x_nodes, y_nodes, x):
    """Evaluate the connect-the-dots interpolant. np.interp does exactly this."""
    return np.interp(x, x_nodes, y_nodes)

In [ ]:
# ---------------------------------------------------------------------------
# Cubic spline: solve for the second derivatives via a tridiagonal system
# ---------------------------------------------------------------------------
# A cubic spline is a cubic on each interval [x_i, x_{i+1}], stitched so that the
# function, its first derivative, and its second derivative are continuous at the
# interior nodes. The unknowns are the second derivatives M_i = s''(x_i). Interior
# continuity gives a tridiagonal linear system for the M_i; the two leftover
# degrees of freedom are fixed by boundary conditions:
#
#   "natural"  : M_0 = M_n = 0            (zero curvature at the ends)
#   "clamped"  : prescribe s'(x_0), s'(x_n)   (here we clamp to the data's slope)

def cubic_spline_moments(x, y, bc="natural"):
    """Return the second-derivative moments M_i defining the cubic spline.

    Parameters
    ----------
    x, y : arrays (n+1,)  sorted nodes and values
    bc   : "natural" or "clamped"

    Returns
    -------
    M : array (n+1,)  second derivatives s''(x_i)
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x) - 1                        # number of intervals
    h = np.diff(x)                        # interval widths h_i = x_{i+1}-x_i

    # Assemble the (n+1)x(n+1) tridiagonal system  A M = d.
    A = np.zeros((n + 1, n + 1))
    d = np.zeros(n + 1)
    # Interior equations: continuity of the first derivative.
    for i in range(1, n):
        A[i, i - 1] = h[i - 1]
        A[i, i]     = 2 * (h[i - 1] + h[i])
        A[i, i + 1] = h[i]
        d[i] = 6 * ((y[i + 1] - y[i]) / h[i] - (y[i] - y[i - 1]) / h[i - 1])

    if bc == "natural":
        A[0, 0] = 1.0; d[0] = 0.0         # M_0 = 0
        A[n, n] = 1.0; d[n] = 0.0         # M_n = 0
    elif bc == "clamped":
        # Clamp end slopes to the secant slope of the first/last panel.
        fp0 = (y[1] - y[0]) / h[0]
        fpn = (y[n] - y[n - 1]) / h[n - 1]
        A[0, 0] = 2 * h[0]; A[0, 1] = h[0]
        d[0] = 6 * ((y[1] - y[0]) / h[0] - fp0)
        A[n, n - 1] = h[n - 1]; A[n, n] = 2 * h[n - 1]
        d[n] = 6 * (fpn - (y[n] - y[n - 1]) / h[n - 1])
    else:
        raise ValueError("bc must be 'natural' or 'clamped'")

    return np.linalg.solve(A, d)


def cubic_spline_eval(x, y, M, xq):
    """Evaluate the cubic spline (given moments M) at query points xq.

    On interval [x_i, x_{i+1}] the spline is the standard moment formula
    combining M_i, M_{i+1} and the endpoint values.
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    xq = np.asarray(xq, float)
    h = np.diff(x)
    # For each query point find its interval index i (clip to a valid panel).
    idx = np.clip(np.searchsorted(x, xq) - 1, 0, len(x) - 2)
    out = np.empty_like(xq)
    for k, xk in enumerate(xq):
        i = idx[k]
        a = x[i + 1] - xk
        b = xk - x[i]
        hi = h[i]
        out[k] = (M[i] * a**3 + M[i + 1] * b**3) / (6 * hi) \
                 + (y[i] / hi - M[i] * hi / 6) * a \
                 + (y[i + 1] / hi - M[i + 1] * hi / 6) * b
    return out

In [ ]:
# ---------------------------------------------------------------------------
# Compare linear vs cubic, natural vs clamped
# ---------------------------------------------------------------------------
def f_demo(x):
    """Target function chosen so the boundary condition affects the ends."""
    return np.cos(x) * np.exp(-0.15 * x)

def show_spline(n_nodes=7, spline="cubic", bc="natural", a=0.0, b=10.0):
    """Sample f at n_nodes points and plot the chosen interpolant against f."""
    x = np.linspace(a, b, n_nodes)
    y = f_demo(x)
    xx = np.linspace(a, b, 600)

    plt.figure()
    plt.plot(xx, f_demo(xx), "k-", lw=1.5, alpha=0.6, label="f(x)")
    if spline == "linear":
        plt.plot(xx, piecewise_linear(x, y, xx), "g-", lw=2,
                 label="piecewise linear")
    else:
        M = cubic_spline_moments(x, y, bc=bc)
        plt.plot(xx, cubic_spline_eval(x, y, M, xx), "r-", lw=2,
                 label=f"cubic spline ({bc})")
    plt.plot(x, y, "bo", ms=6, label="nodes")
    plt.legend(); plt.xlabel("x")
    plt.title(f"{spline} interpolation" + (f", {bc} BC" if spline == "cubic" else ""))
    plt.show()

show_spline(7, "cubic", "natural")

In [ ]:
# ---------------------------------------------------------------------------
# Interactive: node count, linear vs cubic, and the boundary condition
# ---------------------------------------------------------------------------
# With few nodes, switch bc between "natural" and "clamped" and watch how the
# curve bends differently near the two ends, the interior is barely affected.
interact(
    show_spline,
    n_nodes=IntSlider(min=3, max=15, step=1, value=7, description="# nodes"),
    spline=Dropdown(options=["cubic", "linear"], value="cubic", description="type"),
    bc=Dropdown(options=["natural", "clamped"], value="natural", description="cubic BC"),
    a=(-2.0, 2.0, 1.0), b=(6.0, 14.0, 1.0),
);

## Summary

- Cubic splines match value, first derivative, and second derivative at interior nodes, avoiding both kinks and Runge divergence.
- The moments $M_i$ solve a tridiagonal system.
- The boundary condition fixes the two remaining degrees of freedom and shapes the curve near the endpoints while leaving the interior nearly unchanged.
- The linear spline is continuous but has a discontinuous first derivative.